In [110]:
import pandas as pd
from Bio.PDB import PDBParser
import py3Dmol
import numpy as np
from pathlib import Path

# Helper to load a PDB file into a BioPython structure
# If given a path-like object, it opens and parses the file.
def load_pdb(pdb_path):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(Path(pdb_path).stem, str(pdb_path))
    return structure

# Helper to convert probability (0-1) to color (red->yellow->green)
def _prob_to_color(prob):
    """Convert probability (0-1) to hex color: red (0.0) -> yellow (0.5) -> green (1.0)"""
    prob = max(0.0, min(1.0, prob))
    if prob < 0.5:
        r = 255
        g = int(255 * (prob * 2))
        b = 0
    else:
        r = int(255 * (2 * (1 - prob)))
        g = 255
        b = 0
    return f"0x{r:02x}{g:02x}{b:02x}"

def get_residue(structure, residue_id):
    for chain in structure.get_chains():
        for res in chain:
            if res.get_id()[1] == residue_id:
                residue = res
                break
        if residue is not None:
            break
    if residue is None:
        raise ValueError(f"Residue {residue_id} not found in structure.")
    
    return residue

# Helper to show a sphere around the centroid of a residue using py3Dmol.
# residue_id should be the residue number as an integer.
def visualize_residue_ball(pdb_path, residue_id, radius=5.0, style="stick", surface_color="red", opacity=0.3, nearby_style="cartoon", pca=False):
    # Load pdb text in py3Dmol viewer
    with open(pdb_path, "r") as handle:
        pdb_text = handle.read()

    view = py3Dmol.view(width=800, height=600)
    view.addModel(pdb_text, "pdb")
    view.setStyle({"cartoon": {"color": "grey"}})
    view.setBackgroundColor("0xeeeeee")

    structure = load_pdb(pdb_path)
    residue = get_residue(structure, residue_id)

    
    if pca:
        centroid = compute_residue_pca_projection(structure, residue_id)
    else:
        atom_coords = [atom.get_coord() for atom in residue]
        centroid = sum(atom_coords) / len(atom_coords)

    nearby_residue_ids = []
    def _distance(a, b):
        return ((a[0] - b[0])**2 + (a[1] - b[1])**2 + (a[2] - b[2])**2) ** 0.5

    for chain in structure.get_chains():
        for res in chain:
            residue_number = res.get_id()[1]
            if residue_number == residue_id:
                continue
            for atom in res:
                if _distance(atom.get_coord(), centroid) <= radius:
                    nearby_residue_ids.append(residue_number)
                    break

    view.addSphere({
        "center": {"x": float(centroid[0]), "y": float(centroid[1]), "z": float(centroid[2])},
        "radius": float(radius),
        "color": surface_color,
        "opacity": opacity,
    })

    view.addStyle({"resi": [residue_id]}, {style: {"color": "blue"}})
    if nearby_residue_ids:
        view.addStyle({"resi": sorted(set(nearby_residue_ids))}, {nearby_style: {"color": "lime"}})

    view.zoomTo()
    return view

# Helper to add P2Rank pocket centers to a py3Dmol view
def add_pocket_centers(view, prot_id, pocket_radius=2.0, data_dir="../data"):
    """
    Add pocket centers from P2Rank predictions to a py3Dmol view.
    """
    csv_path = Path(data_dir) / "p2rank_output" / f"{prot_id}.pdb_predictions.csv"
    if not csv_path.exists():
        print(f"Warning: Pocket predictions file not found: {csv_path}")
        return view

    pockets_df = pd.read_csv(csv_path, skipinitialspace=True)
    for idx, row in pockets_df.iterrows():
        center_x = float(row["center_x"])
        center_y = float(row["center_y"])
        center_z = float(row["center_z"])
        probability = float(row["probability"])
        color = _prob_to_color(probability)
        view.addSphere({
            "center": {"x": center_x, "y": center_y, "z": center_z},
            "radius": pocket_radius,
            "color": color,
            "opacity": 1,
        })
    return view

# Compute the first two PCA component vectors from atom coordinates
def compute_pca_components(atom_coords, n_components=2):
    """Return the top PCA component vectors for the given atom coordinates."""
    coords = np.asarray(atom_coords, dtype=float)
    if coords.ndim != 2 or coords.shape[1] != 3:
        raise ValueError("atom_coords must be an array of shape (N, 3)")

    centroid = coords.mean(axis=0)
    centered = coords - centroid
    cov = np.dot(centered.T, centered) / max(centered.shape[0] - 1, 1)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    components = eigvecs[:, order[:n_components]].T
    return components

# Add PCA vectors to a py3Dmol view
# The arrows are added after the other geometry so they remain visible.
def add_pca_vectors(view, atom_coords, length=15.0, colors=("cyan", "magenta"), thickness=0.3, origin=None, n_components=2):
    """Add the first two PCA component vectors to the given py3Dmol view."""
    coords = np.asarray(atom_coords, dtype=float)
    if coords.ndim != 2 or coords.shape[1] != 3:
        raise ValueError("atom_coords must be an array of shape (N, 3)")

    if origin is None:
        origin = coords.mean(axis=0)
    else:
        origin = np.asarray(origin, dtype=float)

    components = compute_pca_components(coords, n_components=n_components)
    for component, color in zip(components, colors):
        norm = np.linalg.norm(component)
        if norm == 0:
            continue
        unit_vec = component / norm
        end_pos = origin + unit_vec * length
        start_pos = origin - unit_vec * length
        view.addCylinder({
            "start": {"x": float(start_pos[0]), "y": float(start_pos[1]), "z": float(start_pos[2])},
            "end": {"x": float(end_pos[0]), "y": float(end_pos[1]), "z": float(end_pos[2])},
            "radius": thickness,
            "color": color,
            "opacity": 1.0,
        })
    return view

def compute_point_projection(point, vector, origin=None):
    point = np.asarray(point, dtype=float)
    vector = np.asarray(vector, dtype=float)
    if origin is not None:
        origin = np.asarray(origin, dtype=float)
        point = point - origin

    norm2 = np.dot(vector, vector)
    if norm2 == 0:
        raise ValueError("Projection vector must not be zero-length")

    return np.dot(point, vector) / norm2 * vector

def compute_residue_pca_projection(structure, residue_id, component=0, atom_coords=None):
    if atom_coords is None:
        atom_coords = [atom.get_coord() for atom in structure.get_atoms()]
    point = np.mean([atom.get_coord() for atom in get_residue(structure, residue_id)], axis=0)
    components = compute_pca_components(atom_coords, n_components=component + 1)
    origin = np.mean(atom_coords, axis=0)
    return compute_point_projection(point, components[component], origin=origin) + origin

def add_residue_projection(view, atom_coords, point, color='red', radius=3, component=0):
    components = compute_pca_components(atom_coords, n_components=component + 1)
    origin = np.mean(atom_coords, axis=0)
    proj = compute_point_projection(point, components[component], origin=origin) + origin
    view.addSphere({
            "center": {"x": float(proj[0]), "y": float(proj[1]), "z": float(proj[2])},
            "radius": radius,
            "color": color,
            "opacity": 1,
        })
    
    return view

def add_ball(view, center_coords, color='red', radius=20):
    x, y, z = [float(coord) for coord in center_coords]
    view.addSphere({
            "center": {"x": x,  "y": y, "z": z},
            "radius": radius,
            "color": color,
            "opacity": 0.3,
        })
    
    return view


In [81]:
id = 'Q22011'
view = visualize_residue_ball(f'../data/pdbs/{id}.pdb', radius=25, residue_id=5, nearby_style='cartoon', style='cartoon')
add_pocket_centers(view, id)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [26]:
id = 'Q19975'
view = visualize_residue_ball(f'../data/pdbs/{id}.pdb', radius=10, residue_id=10, nearby_style='cartoon', style='cartoon')
add_pocket_centers(view, id)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [86]:
id = 'Q19975'
view = visualize_residue_ball(f'../data/pdbs/{id}.pdb', radius=20, residue_id=5, nearby_style='cartoon', style='cartoon')
add_pocket_centers(view, id)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [87]:
id = 'Q94072'
view = visualize_residue_ball(f'../data/pdbs/{id}.pdb', radius=25, residue_id=5, nearby_style='cartoon', style='cartoon')
add_pocket_centers(view, id)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [75]:
structure = load_pdb('../data/pdbs/Q19975.pdb')
atom_coords = [atom.get_coord() for atom in structure.get_atoms()]
view = visualize_residue_ball('../data/pdbs/Q19975.pdb', residue_id=10, radius=15)
add_pca_vectors(view, atom_coords, length=60, n_components=1, colors=['cyan'])
view

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
structure = load_pdb('../data/pdbs/Q19975.pdb')
residue_id = 5
atom_coords = [atom.get_coord() for atom in structure.get_atoms()]
view = visualize_residue_ball('../data/pdbs/Q19975.pdb', residue_id=residue_id, radius=15)
add_pca_vectors(view, atom_coords, length=60, n_components=1, colors=['cyan'])
add_residue_projection(view, atom_coords, point=np.mean([atom.get_coord() for atom in get_residue(structure, residue_id)], axis=0), color='magenta')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [122]:
prot_id = 'Q19975'
residue_id = 5
radius = 15
structure = load_pdb(f'../data/pdbs/{prot_id}.pdb')
atom_coords = [atom.get_coord() for atom in structure.get_atoms()]
view = visualize_residue_ball(f'../data/pdbs/{prot_id}.pdb', residue_id=residue_id, radius=radius, pca=True)
add_pocket_centers(view, prot_id)
add_pca_vectors(view, atom_coords, length=60, n_components=1, colors=['cyan'])
add_residue_projection(view, atom_coords, point=np.mean([atom.get_coord() for atom in get_residue(structure, residue_id)], axis=0), color='magenta')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [123]:
prot_id = 'Q94072'
residue_id = 5
radius = 15
structure = load_pdb(f'../data/pdbs/{prot_id}.pdb')
atom_coords = [atom.get_coord() for atom in structure.get_atoms()]
view = visualize_residue_ball(f'../data/pdbs/{prot_id}.pdb', residue_id=residue_id, radius=radius, pca=True)
add_pocket_centers(view, prot_id)
add_pca_vectors(view, atom_coords, length=60, n_components=1, colors=['cyan'])
add_residue_projection(view, atom_coords, point=np.mean([atom.get_coord() for atom in get_residue(structure, residue_id)], axis=0), color='magenta')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.